# DEMO: Star Schemas vs Pre-Joined Tables

When you query base tables directly in Databricks SQL, every JOIN must be **explicit**, which can add complexity to dashboard queries.

In this demo, you'll see:
- How a **star schema** query requires multiple JOINs (and what the execution plan looks like)
- How a **pre-joined Gold table** eliminates JOINs entirely (simpler + often faster)
- How a **view over the star schema** can simplify authoring without removing join work at runtime
- The difference between **BroadcastHashJoin** and **ShuffleExchange** in query plans

> **Note:** Run the setup cell below (Cell 2) to create the demo tables, then open the **SQL Editor** (click "SQL Editor" in the left sidebar) to run the remaining queries there.

---

In [0]:
%run ./Setup

## Part 1: The Star Schema Pattern

Our star schema has a central **fact table** surrounded by **dimension tables**:

```
                dim_customers (50 rows)
                     |
 dim_regions (8) -- fact_sales (3000 rows) -- dim_products (25)
```

**The question:** "Total revenue by region, product category, and customer segment for Q3 2024"

In Databricks SQL, if you query the base tables directly, you write the JOINs yourself.

---

In [0]:
%sql
-- ============================================================
-- Star Schema: Query with multiple JOINs
-- ============================================================
-- This is what dashboard SQL looks like with a star schema.
-- Notice: 3 JOINs required just to get basic dimension attributes.

SELECT
  r.region_name,
  p.category AS product_category,
  c.segment AS customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(f.quantity * f.unit_price * (1 - f.discount_pct)), 2) AS net_revenue
FROM fact_sales f
INNER JOIN dim_regions r ON f.region_id = r.region_id
INNER JOIN dim_products p ON f.product_id = p.product_id
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
WHERE f.sale_date >= '2024-07-01'
  AND f.sale_date < '2024-10-01'
GROUP BY r.region_name, p.category, c.segment
ORDER BY net_revenue DESC;

## What Happens Under the Hood: Shuffles and Broadcasts

When Spark executes a JOIN, it needs data from both sides to be **co-located** (on the same worker). It has two strategies:

| Strategy | How It Works | Performance |
| --- | --- | --- |
| **BroadcastHashJoin** | Copy the small table to ALL workers | Fast — no shuffle of the large table |
| **SortMergeJoin** | Redistribute (shuffle) BOTH tables by the join key | Slow — massive data movement |

With our star schema, the dimension tables are small (8-50 rows), so Spark will likely **broadcast** them. But as dimensions grow or you add more JOINs, performance degrades.

Let's look at the query plan to see this in action:

---

In [0]:
%sql
-- ============================================================
-- EXPLAIN: See the JOINs and data movement
-- ============================================================
-- Look for:
-- - Multiple BroadcastHashJoin operators (one per dimension)
-- - BroadcastExchange operators (copying dim tables to workers)
-- - The scan on fact_sales with DataFilters for the date range

EXPLAIN
SELECT
  r.region_name,
  p.category AS product_category,
  c.segment AS customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(f.quantity * f.unit_price * (1 - f.discount_pct)), 2) AS net_revenue
FROM fact_sales f
INNER JOIN dim_regions r ON f.region_id = r.region_id
INNER JOIN dim_products p ON f.product_id = p.product_id
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
WHERE f.sale_date >= '2024-07-01'
  AND f.sale_date < '2024-10-01'
GROUP BY r.region_name, p.category, c.segment
ORDER BY net_revenue DESC;

## Part 2: The Pre-Joined Table Pattern

A common dashboard-serving pattern is to **pre-join the data upstream** into a wide, flat Gold table. Databricks also recommends star-schema modeling for BI-serving data but pre-joined or pre-aggregated Gold tables are especially useful when you want dashboard-specific simplicity and predictable performance.

**Benefits:**
- Dashboard queries become simple `SELECT ... WHERE ... GROUP BY` (no JOINs)
- The query optimizer has less work to do at runtime
- Dashboard authors don't need to understand the relational model
- Performance is often more predictable (no join strategy decisions at query time)

The **same business question** now becomes a single-table query:

---

In [0]:
%sql
-- ============================================================
-- Pre-Joined Gold Table: Same question, NO JOINs
-- ============================================================
-- Compare the simplicity! Same business answer, but:
-- - No JOINs
-- - No knowledge of the relational model needed
-- - Revenue is pre-calculated (net_revenue column)
-- - Date parts are pre-extracted (sale_quarter)

SELECT
  region_name,
  product_category,
  customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(net_revenue), 2) AS total_revenue
FROM gold_sales_wide
WHERE sale_date >= '2024-07-01'
  AND sale_date < '2024-10-01'
GROUP BY region_name, product_category, customer_segment
ORDER BY total_revenue DESC;

In [0]:
%sql
-- ============================================================
-- EXPLAIN: See how simple the plan is without JOINs
-- ============================================================
-- Compare to the star schema EXPLAIN above:
-- - NO BroadcastHashJoin operators
-- - NO BroadcastExchange operators
-- - Just: Scan -> Filter -> Aggregate -> Sort
-- - This is a much simpler execution plan and is often faster
--   for dashboard-style workloads

EXPLAIN
SELECT
  region_name,
  product_category,
  customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(net_revenue), 2) AS total_revenue
FROM gold_sales_wide
WHERE sale_date >= '2024-07-01'
  AND sale_date < '2024-10-01'
GROUP BY region_name, product_category, customer_segment
ORDER BY total_revenue DESC;

## Part 3: Views — Abstraction Layer, Not Performance Layer

What if you want to keep the star schema for flexibility but still give dashboard authors a simple interface? **Create a view** over the star schema.

A view:
- Maintains the dimensional model underneath (easy to maintain, no data duplication)
- Presents a flat, pre-joined interface to dashboard authors
- Lets you write the JOIN logic once in the view definition instead of repeating it in every query

**Tradeoff:** The JOINs still execute every time the view is queried. For small-to-medium datasets, this is often fine. For very large datasets or many concurrent dashboard users, a materialized view or persisted Gold table may be better.

---

In [0]:
%sql
-- ============================================================
-- Create a VIEW that hides the star schema complexity
-- ============================================================
-- Dashboard authors query this view as if it were a flat table.
-- The JOIN logic lives here, not in every dashboard widget.

CREATE OR REPLACE VIEW v_sales_dashboard AS
SELECT
  f.sale_id,
  f.sale_date,
  YEAR(f.sale_date) AS sale_year,
  QUARTER(f.sale_date) AS sale_quarter,
  c.customer_name,
  c.segment AS customer_segment,
  c.loyalty_tier,
  p.product_name,
  p.category AS product_category,
  p.brand,
  r.region_name,
  r.country,
  f.quantity,
  f.unit_price,
  f.discount_pct,
  ROUND(f.quantity * f.unit_price * (1 - f.discount_pct), 2) AS net_revenue
FROM fact_sales f
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
INNER JOIN dim_products p ON f.product_id = p.product_id
INNER JOIN dim_regions r ON f.region_id = r.region_id;

In [0]:
%sql
-- ============================================================
-- Query the view: dashboard authors see a flat table
-- ============================================================
-- Same business question again, but now the dashboard author
-- doesn't need to know about JOINs or the star schema.
-- The view handles all the complexity.

SELECT
  region_name,
  product_category,
  customer_segment,
  COUNT(*) AS order_count,
  ROUND(SUM(net_revenue), 2) AS total_revenue
FROM v_sales_dashboard
WHERE sale_date >= '2024-07-01'
  AND sale_date < '2024-10-01'
GROUP BY region_name, product_category, customer_segment
ORDER BY total_revenue DESC;

## Part 4: Side-by-Side — Query Complexity Comparison

Look at the three approaches to the same business question:

| Approach | Lines of SQL | JOINs Required | Performance |
| --- | --- | --- | --- |
| **Star Schema** | 15+ | 3 explicit JOINs | Good (broadcast joins on small dims) |
| **Gold Table** | 8 | 0 | Often best for a specific dashboard workload |
| **View** | 8 (view hides 15+) | 0 visible (3 hidden) | Similar to star schema underneath |

**A common Databricks pattern:**
- **Reusable BI serving model:** A star schema in Gold is a common and supported pattern
- **Dashboard-specific serving:** Pre-joined tables, materialized views, or aggregates in Gold can simplify and speed up frequent dashboards
- **Views:** Good for abstraction, but regular views do not precompute the joins

---

In [0]:
%sql
-- ============================================================
-- Dashboard-Ready Queries: Simple and Fast
-- ============================================================
-- These are the kind of queries a dashboard widget should use.
-- No JOINs, clear column names, appropriate filters.

-- Widget 1: Revenue by region (current year)
SELECT region_name, ROUND(SUM(net_revenue), 2) AS total_revenue
FROM gold_sales_wide
WHERE sale_year = 2024
GROUP BY region_name
ORDER BY total_revenue DESC;

-- Widget 2: Top 5 brands by order count
SELECT brand, COUNT(*) AS order_count
FROM gold_sales_wide
WHERE sale_year = 2024
GROUP BY brand
ORDER BY order_count DESC
LIMIT 5;

-- Widget 3: Monthly revenue trend
SELECT sale_month, ROUND(SUM(net_revenue), 2) AS monthly_revenue
FROM gold_sales_wide
WHERE sale_year = 2024
GROUP BY sale_month
ORDER BY sale_month;

---

## Summary

| Concept | Key Takeaway |
| --- | --- |
| **Star Schema** | Works well in Databricks for reusable BI serving, but direct SQL against base tables requires explicit JOINs |
| **Pre-Joined Gold Table** | Eliminates JOINs — simpler queries and often more predictable dashboard performance |
| **View over Star Schema** | Hides complexity while maintaining the dimensional source, but regular views still execute joins at query time |
| **BroadcastHashJoin** | Small dims broadcast to all workers (fast, but still adds plan complexity) |
| **ShuffleExchange** | Data redistributed across workers (expensive — avoid if possible) |


**Practical advice for dashboard design:**
1. Ask for a **Gold table, materialized view, or view** for your dashboard
2. Keep dashboard SQL as simple as possible
3. If your query has repeated JOINs, consider whether they should be abstracted or precomputed upstream
